In [0]:
%python
#Cell 1 — Configuration & Imports
# COMMAND ----------

from pyspark.sql import functions as F
from pyspark.sql.types import StringType, ArrayType
from pyspark.sql.window import Window
from datetime import datetime

# ── Catalog / Schema Configuration ──────────────────────────────────────
CATALOG       = "bfsi"
SILVER_SCHEMA = "silver_layer"
GOLD_SCHEMA   = "gold_layer"

# ── Silver Source Tables (READ-ONLY — no modifications) ────────────────
UNIFIED_TXN_TABLE   = f"{CATALOG}.{SILVER_SCHEMA}.unified_transactions"
FRAUD_ALERTS_TABLE  = f"{CATALOG}.{SILVER_SCHEMA}.fraud_alerts"
DIM_CUSTOMER_RISK   = f"{CATALOG}.{SILVER_SCHEMA}.dim_customer_risk"

# ── Gold Target Table ──────────────────────────────────────────────────
SAR_TABLE = f"{CATALOG}.{GOLD_SCHEMA}.sar_report_feed"

# ── SAR Thresholds ─────────────────────────────────────────────────────
HIGH_VALUE_THRESHOLD = 10_00_000   # ₹10,00,000 (₹10 Lakh)

# ── Job Metadata ───────────────────────────────────────────────────────
GOLD_VERSION   = "v1.0-task3.1"
SCHEDULE_TIME  = "06:50 AM IST (daily)"

print("=" * 70)
print("  GOLD LAYER — SAR REPORT FEED (Task 3.1)")
print("=" * 70)
print(f"  Timestamp        : {datetime.now().isoformat()}")
print(f"  Schedule         : {SCHEDULE_TIME}")
print(f"  SAR Threshold    : ₹{HIGH_VALUE_THRESHOLD:,}")
print(f"  Target Table     : {SAR_TABLE}")
print(f"  Version          : {GOLD_VERSION}")
print(f"  Sources (Silver) :")
print(f"    ├─ {UNIFIED_TXN_TABLE}")
print(f"    ├─ {FRAUD_ALERTS_TABLE}")
print(f"    └─ {DIM_CUSTOMER_RISK}")
print("=" * 70)

# COMMAND ----------
# Create Gold Schema
# COMMAND ----------

spark.sql(f"""
    CREATE SCHEMA IF NOT EXISTS {CATALOG}.{GOLD_SCHEMA}
    COMMENT 'Gold layer — business-ready, compliance feeds, ZERO PII'
""")
print(f"✅ Schema {CATALOG}.{GOLD_SCHEMA} ready.")

In [0]:
# COMMAND ----------
# Cell 2 — Read Silver Source Tables (Read-Only)
# COMMAND ----------

# ══════════════════════════════════════════════════════════════════════════
#  READ SILVER TABLES 
# ══════════════════════════════════════════════════════════════════════════

# ── 1. Unified Transactions (Task 2.2) ─────────────────────────────────
#    All transactions across CARD, UPI, NEFT_RTGS with common schema
df_unified = spark.read.table(UNIFIED_TXN_TABLE)
print(f"📊 Unified transactions  : {df_unified.count():,} records")

# ── 2. Fraud Alerts (Task 2.4) ─────────────────────────────────────────
#    Only suspicious transactions (at least 1 fraud rule triggered)
df_fraud_alerts = spark.read.table(FRAUD_ALERTS_TABLE)
print(f"🚨 Fraud alerts          : {df_fraud_alerts.count():,} records")

# ── 3. Customer Risk Dimension (Task 2.5) ──────────────────────────────
#    SCD2 table — we only need current risk_rating per customer
df_customer_risk = (
    spark.read.table(DIM_CUSTOMER_RISK)
    .filter(F.col("is_current") == True)
    .select("customer_id", "risk_rating")
)
print(f"👤 Active customer risks : {df_customer_risk.count():,} records")

In [0]:
# Cell 3 — Criterion 1: High-Value Transactions (> ₹10,00,000 / day / customer)

# ══════════════════════════════════════════════════════════════════════════
#  CRITERION 1: HIGH-VALUE DAILY TRANSACTIONS
#  Include ALL transactions for any customer whose total daily spend
#  exceeds ₹10,00,000 on any single day.
#
#  Logic:
#    Step 1 — Aggregate per (customer_id, date) and find days above ₹10L
#    Step 2 — Pull ALL individual txns for those (customer, date) combos
# ══════════════════════════════════════════════════════════════════════════

# Step 1: Identify (customer_id, txn_date) pairs where daily sum > ₹10L
df_daily_agg = (
    df_unified
    .withColumn("txn_date", F.to_date("txn_ts"))
    .groupBy("customer_id", "txn_date")
    .agg(F.sum("txn_amount_inr").alias("daily_total"))
    .filter(F.col("daily_total") > HIGH_VALUE_THRESHOLD)
    .select("customer_id", "txn_date")
)

high_value_days = df_daily_agg.count()
print(f"📈 Customer-day combos exceeding ₹{HIGH_VALUE_THRESHOLD:,}: {high_value_days:,}")

# Step 2: Get all individual transactions for these high-value days
df_high_value_txns = (
    df_unified
    .withColumn("txn_date", F.to_date("txn_ts"))
    .join(
        df_daily_agg,
        on=["customer_id", "txn_date"],
        how="inner"
    )
    .select(
        "txn_id",
        "customer_id",
        "txn_channel",
        "txn_amount_inr",
        "txn_ts",
        "txn_date",
        "masked_counterparty",
    )
    # Tag the SAR inclusion reason
    .withColumn("sar_reason", F.lit("HIGH_VALUE_DAILY_THRESHOLD"))
    # No fraud rule for purely high-value picks
    .withColumn("fraud_rule_triggered", F.lit(None).cast(ArrayType(StringType())))
)

high_value_count = df_high_value_txns.count()
print(f"💰 High-value transactions (all txns on those days): {high_value_count:,}")


In [0]:
# Cell 4 — Criterion 2: Fraud-Flagged Transactions (Task 2.4 Rules)

# COMMAND ----------

# ══════════════════════════════════════════════════════════════════════════
#  CRITERION 2: FRAUD-FLAGGED TRANSACTIONS
#  All transactions flagged by at least one fraud rule in Task 2.4
#  Source: bfsi.silver_layer.fraud_alerts (already filtered to suspicious)
# ══════════════════════════════════════════════════════════════════════════

df_fraud_txns = (
    df_fraud_alerts
    .withColumn("txn_date", F.to_date("txn_ts"))
    .select(
        "txn_id",
        "customer_id",
        "txn_channel",
        "txn_amount_inr",
        "txn_ts",
        "txn_date",
        "masked_counterparty",
        "fraud_rule_triggered",
    )
    # Tag the SAR inclusion reason
    .withColumn("sar_reason", F.lit("FRAUD_RULE_TRIGGERED"))
)

fraud_count = df_fraud_txns.count()
print(f"🚨 Fraud-flagged transactions for SAR: {fraud_count:,}")


In [0]:
# Cell 5 — Union & Deduplicate

# COMMAND ----------

# ══════════════════════════════════════════════════════════════════════════
#  UNION BOTH CRITERIA & DEDUPLICATE BY txn_id
#  A transaction can qualify under BOTH criteria (high-value + fraud).
#  In such cases, we keep ONE row and prefer the fraud-flagged version
#  (to preserve the fraud_rule_triggered array).
# ══════════════════════════════════════════════════════════════════════════

# Union both sets
df_combined = df_high_value_txns.unionByName(df_fraud_txns)

print(f"📋 Combined (before dedup): {df_combined.count():,}")
print(f"   ├─ High-value : {high_value_count:,}")
print(f"   └─ Fraud-flag : {fraud_count:,}")

# Deduplicate: keep one row per txn_id
# Priority: fraud-flagged row wins (has non-null fraud_rule_triggered)
dedup_window = Window.partitionBy("txn_id").orderBy(
    # Rows with fraud_rule_triggered NOT NULL come first
    F.when(F.col("fraud_rule_triggered").isNotNull(), F.lit(0)).otherwise(F.lit(1)).asc()
)

df_deduped = (
    df_combined
    .withColumn("_rn", F.row_number().over(dedup_window))
    .filter(F.col("_rn") == 1)
    .drop("_rn")
)

deduped_count = df_deduped.count()
duplicates_removed = df_combined.count() - deduped_count
print(f"\n✅ After dedup: {deduped_count:,} unique transactions")
print(f"   Duplicates merged: {duplicates_removed:,}")


In [0]:
# Cell 6 — Join Customer Risk Rating (SCD2 Dimension)

# COMMAND ----------

# ══════════════════════════════════════════════════════════════════════════
#  JOIN with dim_customer_risk (SCD2 — current version only)
#  Provides risk_rating per customer for the SAR report
# ══════════════════════════════════════════════════════════════════════════

df_with_risk = (
    df_deduped
    .join(
        df_customer_risk,
        on="customer_id",
        how="left"
    )
    # Fill missing risk ratings with UNKNOWN
    .withColumn(
        "risk_rating",
        F.coalesce(F.col("risk_rating"), F.lit("UNKNOWN"))
    )
)

# Risk rating distribution
print("📊 Risk rating distribution in SAR feed:")
display(
    df_with_risk
    .groupBy("risk_rating")
    .agg(F.count("*").alias("txn_count"))
    .orderBy("risk_rating")
)


In [0]:
# Cell 7 — Build Final SAR Report (ZERO PII)

# COMMAND ----------

# ══════════════════════════════════════════════════════════════════════════
#  BUILD FINAL SAR REPORT FEED — ZERO PII
#
#  All identifiers are tokenized with SHA-256:
#    - txn_id       → masked_txn_id   (SHA-256 hash, irreversible)
#    - customer_id  → customer_token  (SHA-256 hash, irreversible)
#
#  reporting_branch: Derived deterministically from the masked_counterparty
#    and txn_channel using a hash-based branch code mapping.
#    In production, this would be joined from a branch master table.
# ══════════════════════════════════════════════════════════════════════════

# Define branch code pools for deterministic assignment
_BRANCH_CODES = [
    "BR-MUM-001", "BR-DEL-002", "BR-BLR-003", "BR-CHN-004", "BR-HYD-005",
    "BR-KOL-006", "BR-PUN-007", "BR-AHM-008", "BR-JAI-009", "BR-LKO-010",
    "BR-GUW-011", "BR-PAT-012", "BR-BHP-013", "BR-CHA-014", "BR-THI-015",
    "BR-NGP-016", "BR-IND-017", "BR-VIZ-018", "BR-SUR-019", "BR-COI-020",
]

NUM_BRANCHES = len(_BRANCH_CODES)

df_sar_report = (
    df_with_risk
    .select(
        # ── report_date: date of report generation ──
        F.current_date().alias("report_date"),

        # ── masked_txn_id: SHA-256 tokenized transaction ID ──
        F.sha2(F.col("txn_id").cast("string"), 256).alias("masked_txn_id"),

        # ── customer_token: SHA-256 tokenized customer identifier ──
        F.sha2(F.col("customer_id").cast("string"), 256).alias("customer_token"),

        # ── txn_amount: amount in INR ──
        F.round(F.col("txn_amount_inr"), 2).alias("txn_amount"),

        # ── txn_channel: CARD / UPI / NEFT_RTGS ──
        F.col("txn_channel"),

        # ── fraud_rule_triggered: array of triggered rules ──
        # NULL/empty for transactions included solely due to high-value threshold
        F.coalesce(
            F.col("fraud_rule_triggered"),
            F.array().cast(ArrayType(StringType()))
        ).alias("fraud_rule_triggered"),

        # ── risk_rating: from SCD2 dim_customer_risk ──
        F.col("risk_rating"),

        # ── reporting_branch: deterministic hash-based branch code ──
        # Uses CRC32 of customer_id + txn_channel for consistent mapping
        F.element_at(
            F.array(*[F.lit(b) for b in _BRANCH_CODES]),
            (F.abs(F.crc32(F.concat(F.col("customer_id"), F.col("txn_channel")))) % NUM_BRANCHES + 1).cast("int")
        ).alias("reporting_branch"),
    )
)

print(f"✅ SAR report built: {df_sar_report.count():,} records")
print("\n📋 Schema (ZERO PII):")
df_sar_report.printSchema()


In [0]:
# Cell 8 — PII Absence Verification

# COMMAND ----------

# ══════════════════════════════════════════════════════════════════════════
#  PII ABSENCE VERIFICATION
#  Validates that ZERO PII exists in the final SAR report
# ══════════════════════════════════════════════════════════════════════════

# ── Allowed columns (ZERO PII) ──
ALLOWED_COLUMNS = {
    "report_date", "masked_txn_id", "customer_token", "txn_amount",
    "txn_channel", "fraud_rule_triggered", "risk_rating", "reporting_branch"
}

# ── PII column names that must NOT appear ──
PII_BLACKLIST = {
    "customer_id", "txn_id", "card_number", "pan_number",
    "mobile_number", "email", "full_name", "ip_address",
    "debtor_name", "creditor_name", "payer_vpa", "payee_vpa",
    "masked_counterparty", "masked_instrument"
}

actual_columns = set(df_sar_report.columns)

# Check 1: Only allowed columns present
extra_columns = actual_columns - ALLOWED_COLUMNS
if extra_columns:
    raise ValueError(f"❌ PII CHECK FAILED: Unexpected columns found: {extra_columns}")
else:
    print("✅ Check 1: Only allowed columns present")

# Check 2: No PII columns leaked
pii_leaked = actual_columns & PII_BLACKLIST
if pii_leaked:
    raise ValueError(f"❌ PII CHECK FAILED: PII columns found: {pii_leaked}")
else:
    print("✅ Check 2: No PII columns in output")

# Check 3: All required columns present
missing_columns = ALLOWED_COLUMNS - actual_columns
if missing_columns:
    raise ValueError(f"❌ SCHEMA CHECK FAILED: Missing columns: {missing_columns}")
else:
    print("✅ Check 3: All 8 required columns present")

# Check 4: masked_txn_id and customer_token are proper SHA-256 hashes (64 hex chars)
sample_row = df_sar_report.select("masked_txn_id", "customer_token").first()
if sample_row:
    assert len(sample_row["masked_txn_id"]) == 64, "masked_txn_id is not 64-char SHA-256"
    assert len(sample_row["customer_token"]) == 64, "customer_token is not 64-char SHA-256"
    print("✅ Check 4: Tokenized IDs are valid SHA-256 (64 hex characters)")

print("\n🔒 PII VERIFICATION: ALL CHECKS PASSED — ZERO PII in output")


In [0]:
# Cell 9 — Write SAR Report to Gold Delta Table

# COMMAND ----------

# ══════════════════════════════════════════════════════════════════════════
#  WRITE SAR REPORT FEED TO GOLD DELTA TABLE
#  Mode: overwrite (daily snapshot — compliance system consumes full report)
#  Schedule: Daily refresh by 06:50 AM IST
# ══════════════════════════════════════════════════════════════════════════

# Add Gold layer metadata
df_final = (
    df_sar_report
    .withColumn("_gold_load_ts", F.current_timestamp())
    .withColumn("_gold_version", F.lit(GOLD_VERSION))
    .withColumn("_refresh_schedule", F.lit(SCHEDULE_TIME))
)

# Write to Gold Delta table
df_final.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(SAR_TABLE)

print(f"\n✅ Gold table written: {SAR_TABLE}")
print(f"   Records: {spark.read.table(SAR_TABLE).count():,}")


In [0]:
# Cell 10 — Schedule Configuration (Databricks Job)

# COMMAND ----------

# ══════════════════════════════════════════════════════════════════════════
#  SCHEDULE CONFIGURATION
#  This notebook should be scheduled as a Databricks Job with:
#    Cron Expression : 50 1 * * *     (06:50 AM IST = 01:20 UTC)
#    Timezone        : Asia/Kolkata
#    Retry Policy    : 2 retries, 5-minute intervals
#    Alert           : Email on failure to compliance-team@bank.com
#
#  Databricks Job Setup (via UI or API):
#  {
#    "name": "Gold_SAR_Report_Feed_Daily",
#    "schedule": {
#      "quartz_cron_expression": "0 50 1 * * ?",
#      "timezone_id": "Asia/Kolkata"
#    },
#    "tasks": [{
#      "task_key": "sar_report_feed",
#      "notebook_task": {
#        "notebook_path": "/Gold_Layer/3.1.SAR_REPORT_FEED"
#      }
#    }],
#    "max_retries": 2,
#    "min_retry_interval_millis": 300000
#  }
# ══════════════════════════════════════════════════════════════════════════

print("📅 Schedule Configuration:")
print(f"   Cron     : 0 50 1 * * ?  (UTC) = 06:50 AM IST daily")
print(f"   Timezone : Asia/Kolkata")
print(f"   Retries  : 2 (5-minute intervals)")
print(f"   Alert    : On failure → compliance team")


In [0]:

# ══════════════════════════════════════════════════════════════════════════
#  TASK 3.1 — SAR REPORT FEED — COMPLETION SUMMARY
# ══════════════════════════════════════════════════════════════════════════

df_result = spark.read.table(SAR_TABLE)

print("=" * 70)
print("  TASK 3.1: SAR REPORT FEED — COMPLETE")
print("=" * 70)

# ── Record counts ──
total_records = df_result.count()
print(f"\n📊 Total SAR records: {total_records:,}")

# ── By channel ──
print("\n📡 By Transaction Channel:")
display(
    df_result
    .groupBy("txn_channel")
    .agg(
        F.count("*").alias("record_count"),
        F.round(F.sum("txn_amount"), 2).alias("total_amount_inr"),
        F.round(F.avg("txn_amount"), 2).alias("avg_amount_inr"),
    )
    .orderBy("txn_channel")
)

# ── By risk rating ──
print("\n⚠️ By Risk Rating:")
display(
    df_result
    .groupBy("risk_rating")
    .agg(
        F.count("*").alias("record_count"),
        F.round(F.sum("txn_amount"), 2).alias("total_amount_inr"),
    )
    .orderBy("risk_rating")
)

# ── Fraud rules breakdown ──
fraud_flagged = df_result.filter(F.size("fraud_rule_triggered") > 0).count()
high_value_only = total_records - fraud_flagged
print(f"\n🔍 SAR Inclusion Breakdown:")
print(f"   Fraud rule triggered  : {fraud_flagged:,}")
print(f"   High-value threshold  : {high_value_only:,} (threshold-only, no fraud rule)")
print(f"   Total                 : {total_records:,}")

# ── Schema validation ──
expected = [
    "report_date", "masked_txn_id", "customer_token", "txn_amount",
    "txn_channel", "fraud_rule_triggered", "risk_rating", "reporting_branch"
]
missing = [c for c in expected if c not in df_result.columns]
print(f"\n📋 Schema: {'✅ ALL 8 required columns present' if not missing else f'❌ MISSING: {missing}'}")

# ── PII check ──
pii_cols = {"customer_id", "txn_id", "card_number", "pan_number",
            "mobile_number", "email", "full_name", "ip_address"}
leaked = set(df_result.columns) & pii_cols
print(f"🔒 PII Check: {'✅ ZERO PII — compliant' if not leaked else f'❌ PII FOUND: {leaked}'}")

# ── Sample output ──
print("\n📄 Sample SAR records (first 10):")
display(
    df_result
    .select(*expected)
    .orderBy(F.col("txn_amount").desc())
    .limit(10)
)

print(f"\n   Gold Version : {GOLD_VERSION}")
print(f"   Schedule     : {SCHEDULE_TIME}")
print(f"   Timestamp    : {datetime.now().isoformat()}")
print("=" * 70)
